In [ ]:
#four-class design
#Ingredient
#Recipe
#RecipeBook
#RecipeApp or RecipeGUI
#1.Ingredient
#Represents a fixed record: name, quantity, unit. It should support scaling.
class Ingredient:
    def __init__(self, name: str, quantity: float, unit: str):
        self.name = name
        self.quantity = quantity
        self.unit = unit

    def scaled(self, factor: float) -> "Ingredient":
        return Ingredient(self.name, self.quantity * factor, self.unit)

    def to_tuple(self) -> tuple[str, float, str]:
        return (self.name, self.quantity, self.unit)


In [ ]:
#2.Recipe: stores ingredients and steps in lists.
class Recipe:
    def __init__(self, title: str, servings: int):
        self.title = title
        self.servings = servings
        self.ingredients: list[Ingredient] = []
        self.steps: list[str] = []

    def add_ingredient(self, ingredient: Ingredient) -> None:
        self.ingredients.append(ingredient)

    def add_step(self, step: str) -> None:
        self.steps.append(step)

    def scaled_ingredients(self, new_servings: int) -> list[Ingredient]:
        if self.servings <= 0:
            raise ValueError("Servings must be positive")
        factor = new_servings / self.servings
        return [ing.scaled(factor) for ing in self.ingredients]

In [ ]:
#3. RecipeBook:Manages all recipes. Use a dictionary keyed by title.
class RecipeBook:
    def __init__(self):
        self._recipes: dict[str, Recipe] = {}

    def add(self, recipe: Recipe) -> None:
        self._recipes[recipe.title] = recipe

    def get(self, title: str) -> Recipe | None:
        return self._recipes.get(title)

    def list_titles(self) -> list[str]:
        return list(self._recipes.keys())

    def generate_shopping_list(
        self, titles: list[str]
    ) -> list[tuple[str, float, str]]:
        totals: dict[tuple[str, str], float] = {}

        for title in titles:
            recipe = self._recipes.get(title)
            if recipe is None:
                continue
            for ing in recipe.ingredients:
                key = (ing.name, ing.unit)
                totals[key] = totals.get(key, 0) + ing.quantity

        return [
            (name, quantity, unit)
            for (name, unit), quantity in totals.items()
        ]

In [ ]:
#4. RecipeApp / GUI: A minimal Tkinter skeleton:
import tkinter as tk
from tkinter import messagebox


class RecipeApp(tk.Tk):
    def __init__(self, book: RecipeBook):
        super().__init__()
        self.book = book
        self.title("Recipe Manager")

        self.listbox = tk.Listbox(self, selectmode=tk.EXTENDED)
        self.listbox.pack(fill=tk.BOTH, expand=True)

        tk.Button(self, text="Refresh", command=self.refresh).pack()
        tk.Button(self, text="View Selected", command=self.view_selected).pack()
        tk.Button(
            self, text="Generate Shopping List", command=self.shopping_selected
        ).pack()

        self.refresh()

    def refresh(self) -> None:
        self.listbox.delete(0, tk.END)
        for title in self.book.list_titles():
            self.listbox.insert(tk.END, title)

    def view_selected(self) -> None:
        selection = self.listbox.curselection()
        if not selection:
            return
        title = self.listbox.get(selection[0])
        recipe = self.book.get(title)
        if recipe is None:
            return
        text = f"{recipe.title} ({recipe.servings} servings)\n\n"
        text += "Ingredients:\n"
        for ing in recipe.ingredients:
            text += f"- {ing.quantity} {ing.unit} {ing.name}\n"
        text += "\nSteps:\n"
        for i, step in enumerate(recipe.steps, 1):
            text += f"{i}. {step}\n"
        messagebox.showinfo(title, text)

    def shopping_selected(self) -> None:
        titles = [self.listbox.get(i) for i in self.listbox.curselection()]
        items = self.book.generate_shopping_list(titles)
        text = "\n".join(f"{name}: {qty} {unit}" for name, qty, unit in items)
        messagebox.showinfo("Shopping List", text or "No items selected.")

In [ ]:
#. UML diagram
RecipeApp/GUI --> RecipeBook
RecipeBook --> Recipe
Recipe --> Ingredient
pip install pylint
pyreverse -o png -p RecipeManager your_project_folder/

In [ ]:
#sample output
=== Recipe Manager ===
1. Add recipe
2. List recipes
3. View recipe
4. Scale recipe
5. Generate shopping list
0. Quit

In [ ]:
#sample shopping list:
Shopping List:
- Tomato: 700 g
- Onion: 2 count
- Olive oil: 30 ml